In [1]:
import cv2
import glob
import os
from ultralytics import YOLO

# 1. Configuración inicial
ruta_pesos = '../models/entrenamiento_script_estable/weights/best.pt'
model = YOLO(ruta_pesos)
nombres_clases = {0: 'Vehiculo', 1: 'Peaton', 2: 'Dos_Ruedas', 3: 'Senales'}

# Crear carpeta para los resultados visuales si no existe
os.makedirs('../reports/alertas_visuales', exist_ok=True)

# 2. Tomar una imagen de prueba
imagen_ruta = glob.glob('../data/enhanced/images/test/*.jpg')[0]
imagen_cv = cv2.imread(imagen_ruta)
img_alto, img_ancho = imagen_cv.shape[:2]
area_total = img_ancho * img_alto

# 3. Hacer predicción
resultados = model(imagen_ruta, verbose=False)

# 4. Procesar y dibujar
for resultado in resultados:
    cajas = resultado.boxes
    for caja in cajas:
        confianza = float(caja.conf[0])
        if confianza < 0.4:
            continue
            
        clase_id = int(caja.cls[0])
        nombre_objeto = nombres_clases[clase_id]
        x1, y1, x2, y2 = map(int, caja.xyxy[0].tolist())
        
        area_obj = (x2 - x1) * (y2 - y1)
        porcentaje = (area_obj / area_total) * 100
        
        # Determinar color (BGR en OpenCV) y texto según el riesgo
        color = (0, 255, 0) # Verde por defecto (Bajo)
        texto_alerta = f"{nombre_objeto} {porcentaje:.1f}%"
        
        if clase_id == 0 and porcentaje > 15.0:
            color = (0, 0, 255) # Rojo (ALTO)
            texto_alerta = "¡COLISION INMINENTE!"
        elif clase_id in [1, 2] and porcentaje > 5.0:
            color = (0, 0, 255) # Rojo (CRÍTICO)
            texto_alerta = "¡PELIGRO PEATON/MOTO!"
        elif porcentaje > 8.0:
            color = (0, 255, 255) # Amarillo (MEDIO)
            texto_alerta = "Precaucion"

        # Dibujar la caja y el texto en la imagen
        cv2.rectangle(imagen_cv, (x1, y1), (x2, y2), color, 2)
        
        # Fondo negro para el texto para que resalte
        (w, h), _ = cv2.getTextSize(texto_alerta, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 1)
        cv2.rectangle(imagen_cv, (x1, y1 - 20), (x1 + w, y1), color, -1)
        cv2.putText(imagen_cv, texto_alerta, (x1, y1 - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 0), 1)

# 5. Guardar la imagen final
nombre_salida = os.path.basename(imagen_ruta)
ruta_guardado = f"../reports/alertas_visuales/alerta_{nombre_salida}"
cv2.imwrite(ruta_guardado, imagen_cv)

print(f"¡Imagen de alerta generada con éxito en: {ruta_guardado}!")

¡Imagen de alerta generada con éxito en: ../reports/alertas_visuales/alerta_cabc30fc-e7726578.jpg!
